# Pixel-wise runoff-onset vs climate anomaly correlations (exploratory)

Correlates the runoff-onset anomaly against each ERA5-Land variable's anomaly per
pixel across water years, on an Equal-Earth grid **derived on the fly**
(`era5.combined_anomaly_eqearth`: ERA5 stack → EPSG:8857, coarse onset from the
public pyramid, masked + merged, anomalies vs the all-year median). The standing
eqearth stores this used to read are retired. Output: the correlations zarr at
`era5.correlations_store_path(config)` (local, under `scratch/`) — this notebook's own product,
read by nothing else.

In [ ]:
import xarray as xr
from global_snowmelt_runoff_onset.config import Config
from gsro_analysis import era5, settings
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
config = settings.load_config()  # the dataset version lives in settings.CONFIG_FILE

In [ ]:
# a local dask cluster (the only compute this repository uses besides GitHub Actions):
from dask.distributed import LocalCluster
client = LocalCluster(n_workers=4, threads_per_worker=2, memory_limit='2.5GB').get_client()   # sized for a 16 GB box


In [ ]:
def get_era5_correlations(config):

    def calculate_correlations(ds):
        """Calculate correlations between runoff_onset and all climate variables"""
        
        # Get the runoff_onset data
        runoff_data = ds['runoff_onset']
        
        # Initialize list to store correlation results
        corr_data_vars = {}
        
        # Variables that have month dimension
        monthly_vars = [var for var in ds.data_vars if 'month' in ds[var].dims and var != 'runoff_onset']
        
        # Variables that don't have month dimension (excluding runoff_onset itself)
        non_monthly_vars = [var for var in ds.data_vars if 'month' not in ds[var].dims and var != 'runoff_onset']
        
        print(f"Processing monthly variables: {monthly_vars}")
        print(f"Processing non-monthly variables: {non_monthly_vars}")
        
        # Calculate correlations for monthly variables
        for var in monthly_vars:
            print(f"Calculating correlation for {var}...")
            # Use xarray's corr function along water_year dimension
            corr_data_vars[var] = xr.corr(runoff_data, ds[var], dim='water_year')
        
        # Calculate correlations for non-monthly variables
        for var in non_monthly_vars:
            print(f"Calculating correlation for {var}...")
            corr_data_vars[var] = xr.corr(runoff_data, ds[var], dim='water_year')
            # Add month dimension by broadcasting
            corr_data_vars[var] = corr_data_vars[var].expand_dims({'month': ds.month})
        
        # Create the correlation dataset
        corr_ds = xr.Dataset(corr_data_vars)
        
        # Copy coordinates
        corr_ds = corr_ds.assign_coords({
            'month': ds.month,
            'y': ds.y,
            'x': ds.x,
            'spatial_ref': ds.spatial_ref
        })
        
        return corr_ds
    
    # derived on the fly — the standing eqearth stores are retired (2026-08-24)
    combined_runoff_and_era5_10yr_anomaly_ds = era5.combined_anomaly_eqearth(config)
    
    runoff_and_climate_correlations_ds = calculate_correlations(combined_runoff_and_era5_10yr_anomaly_ds)
    
    # uniform chunks: the on-the-fly Equal-Earth derivation leaves ragged dask chunks that zarr refuses
    runoff_and_climate_correlations_ds = runoff_and_climate_correlations_ds.chunk({'month': -1, 'y': 512, 'x': 512})
    for name in list(runoff_and_climate_correlations_ds.variables):   # drop the source stores' (zarr v2) codecs before a zarr v3 write
        runoff_and_climate_correlations_ds[name].encoding = {}
    runoff_and_climate_correlations_ds.to_zarr(
        era5.correlations_store_path(config), mode='w')   # local, under scratch/
    
    return 'success'
    

In [ ]:
# run in this process: the chunked xarray/dask work inside spreads over the local cluster's workers
# (submitting the whole function as ONE task to a worker was the old remote-cluster pattern and
# kills a 2.5 GB worker)
result = get_era5_correlations(config)
result

In [ ]:
# the correlations zarr is now at era5.correlations_store_path(config) (local, under scratch/)
print(era5.correlations_store_path(config))

In [ ]:
# derived on the fly — the standing eqearth stores are retired (2026-08-24)
combined_runoff_and_era5_10yr_anomaly_ds = era5.combined_anomaly_eqearth(config)
combined_runoff_and_era5_10yr_anomaly_ds

In [ ]:
combined_runoff_and_era5_10yr_anomaly_ds['temperature_2m'].sel(month='spring_month_1').compute().plot.imshow(col='water_year', col_wrap=5,vmin=-10,vmax=10,cmap='RdBu_r')

In [ ]:
runoff_onset_and_era5_10yr_anomaly_correlations_ds = xr.open_zarr(
    era5.correlations_store_path(config),   # local, under scratch/
    decode_coords='all', chunks='auto')
runoff_onset_and_era5_10yr_anomaly_correlations_ds

In [ ]:
runoff_onset_and_era5_10yr_anomaly_correlations_ds = runoff_onset_and_era5_10yr_anomaly_correlations_ds.compute()
runoff_onset_and_era5_10yr_anomaly_correlations_ds

In [ ]:
runoff_onset_and_era5_10yr_anomaly_correlations_ds['temperature_2m'].plot.imshow(col='month',col_wrap=3,cmap='RdBu')

In [ ]:
#WUS_bbox = minx=-130, miny=30, maxx=-60, maxy=75
WUS_bbox = [-130, 30, -60, 75]
HMA_bbox = [65, 25, 110, 45]
northern_europe_and_asia_bbox = [-10, 45, 80, 75]
bbox = HMA_bbox
region_correlations_ds = runoff_onset_and_era5_10yr_anomaly_correlations_ds.rio.clip_box(*bbox, crs="EPSG:4326")
region_correlations_ds

In [ ]:
# don't need surface_net_solar_radiation_sum or surface_net_thermal_radiation_sum
# with RdBu colormap, red means increase in var means earlier runoff onset, blue means increase in var means later runoff onset

In [ ]:
import easysnowdata
import pandas as pd

In [ ]:
easysnowdata.utils.datetime_to_DOWY(pd.Timestamp('2024-03-01'))

In [ ]:
easysnowdata.utils.datetime_to_DOWY(pd.Timestamp('2024-06-01'))

In [ ]:
easysnowdata.utils.datetime_to_DOWY(pd.Timestamp('2024-07-01'))

In [ ]:
import cartopy.crs as ccrs


In [ ]:


for var in region_correlations_ds.data_vars:
    print(var)
    fig = region_correlations_ds[var].plot.imshow(col='month',col_wrap=3,cmap='RdBu',sharex=True,sharey=True,subplot_kws={'projection': ccrs.EqualEarth()},figsize=(12,8))
    for ax in fig.axs.flatten():
        ax.set_aspect('equal')
        ax.set_title(ax.get_title().replace('month = ',''))
        ax.gridlines(draw_labels=False)
        #ctx.add_basemap(ax,crs=ccrs.EqualEarth())
    fig.fig.suptitle(f'Correlation between 10-year runoff onset anomaly and\n10-year anomaly in ERA5-Land {var}', y=1.00)
#     plt.show()
# fig = region_correlations_ds[var].plot.imshow(col='month',col_wrap=3,cmap='RdBu',sharex=True,sharey=True,subplot_kws={'projection': ccrs.EqualEarth()},figsize=(12,8))
# for ax in fig.axs.flatten():
#     ax.set_aspect('equal')
#     ax.set_title(ax.get_title().replace('month = ',''))
#     ax.gridlines(draw_labels=False)
#     #ctx.add_basemap(ax,crs=ccrs.EqualEarth())
# fig.fig.suptitle(f'Correlation between 10-year runoff onset anomaly and\n10-year anomaly in ERA5-Land {var}', y=1.00)

In [ ]:
url = (f"https://data.earthenv.org/mountains/standard/GMBA_Inventory_v2.0_standard_300.zip")
gmba_gdf = gpd.read_file("zip+" + url)
gmba_gdf

In [ ]:
mountain_range_name = "Sierra Nevada"
mountain_range_name = "Olympic Mountains"
mountain_range_name = "Brooks Range"

In [ ]:
mountain_range_gdf = gmba_gdf[gmba_gdf['MapName']==mountain_range_name]
mountain_range_gdf

In [ ]:
mountain_range_correlations_ds = runoff_onset_and_era5_10yr_anomaly_correlations_ds.rio.clip(mountain_range_gdf.to_crs(runoff_onset_and_era5_10yr_anomaly_correlations_ds.rio.crs).geometry)
mountain_range_correlations_ds

In [ ]:
mountain_range_correlations_ds['surface_net_solar_radiation_sum'].plot.imshow(col='month',col_wrap=6)

In [ ]:
mountain_range_correlations_ds['surface_solar_radiation_downwards_sum'].plot.imshow(col='month',col_wrap=6)

In [ ]:

var = 'temperature_2m'
fig = mountain_range_correlations_ds[var].plot.imshow(col='month',col_wrap=6,vmin=-1,vmax=1,cmap='RdBu_r',figsize=(20,5))
for i, ax in enumerate(fig.axs.flatten()):
    mountain_range_gdf.to_crs(mountain_range_correlations_ds.rio.crs).boundary.plot(ax=ax, color='black', linewidth=1)
    avg_correlation = mountain_range_correlations_ds[var].mean(dim=['x','y'])
    month = ax.get_title().split("=")[-1]
    ax.set_title(f'{month}\navg_corr={avg_correlation.values[i]:.2f}')
    ax.set_aspect('equal')
    ax.axis('off')

In [ ]:
for var in mountain_range_correlations_ds.data_vars:
    fig = mountain_range_correlations_ds[var].plot.imshow(col='month',col_wrap=6,vmin=-1,vmax=1,cmap='RdBu_r',figsize=(12,5))
    for i, ax in enumerate(fig.axs.flatten()):
        mountain_range_gdf.to_crs(mountain_range_correlations_ds.rio.crs).boundary.plot(ax=ax, color='black', linewidth=1)
        avg_correlation = mountain_range_correlations_ds[var].mean(dim=['x','y'])
        month = ax.get_title().split("=")[-1]
        ax.set_title(f'{month}\navg_corr={avg_correlation.values[i]:.2f}')
        ax.axis('off')
    fig.fig.suptitle(var,y=1.02)

In [ ]:
# derived on the fly — the standing eqearth stores are retired (2026-08-24)
combined_runoff_and_era5_10yr_anomaly_ds = era5.combined_anomaly_eqearth(config)
combined_runoff_and_era5_10yr_anomaly_ds

In [ ]:
mountain_range_mean_anomalies = combined_runoff_and_era5_10yr_anomaly_ds.rio.clip(mountain_range_gdf.to_crs(combined_runoff_and_era5_10yr_anomaly_ds.rio.crs).geometry).mean(['x','y'])
mountain_range_mean_anomalies

In [ ]:
mountain_range_mean_anomalies.sel(month='spring_month_1').plot.scatter(x='temperature_2m', y='runoff_onset')

In [ ]:
mountain_range_mean_anomalies['runoff_onset'].plot(ax=ax)
mountain_range_mean_anomalies['temperature_2m'].sel(month='spring_month_1')

In [ ]:
for month in mountain_range_mean_anomalies['month'].values:
    corr = xr.corr(mountain_range_mean_anomalies['temperature_2m'].sel(month=month), mountain_range_mean_anomalies['runoff_onset']).compute().values
    print(f'Correlation for {month}: {corr}')
    

In [ ]:
corr = xr.corr(mountain_range_mean_anomalies['temperature_2m'].sel(month=['spring_month_1', 'spring_month_2','spring_month_3']).mean(dim='month'), sierras_mean_anomalies['runoff_onset']).compute().values
print(f'Correlation for spring months: {corr}')
